In [2]:
# !pip install --upgrade pip
!pip install qdrant-client transformers torch torchaudio librosa numpy musicbrainzngs
pip install syncedlyrics "qdrant-client[fastembed]>=1.14.2" mutagen

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.7 MB/s eta 0:00:00


In [1]:
from file_processor import FileProcessor
from search_engine import LyricsDB

from qdrant_client import QdrantClient
import torch
import laion_clap

In [2]:
qd_client = QdrantClient("http://localhost:6333")
fp = FileProcessor()

In [3]:
folder = r"S:\Test"
# data = fp.process_folder(folder, better_lyrics_quality=True)
# fp.to_json('test.json')
fp.load('test.json')
data = fp.to_dict()
# fp.enhance_lyrics()

## DB creation

In [7]:
db = LyricsDB(qdrant_client=qd_client, collection_name="qwen", model_name='Qwen/Qwen3-Embedding-0.6B')
# db.fit(data, folder)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [8]:
db.search("lucky", limit=3)

[ScoredPoint(id='ec261d06-4337-4841-a968-0963c81ee595', version=10, score=0.5, payload={'lyrics': "Lucky love belongs in teenage heaven\nI know, I know\n'Cause I've been there with you tonight\nLucky love belongs in teenage heaven\nI know, I know\nI'm a prisoner of hope tonight\n\nI believe life could be paradise once again\nAnd the love we thought we lost is sleeping within\nClose your eyes, it's something for you\n\nLucky love belongs in teenage heaven\nI know, I know\n'Cause I've been there with you tonight\nLucky love belongs in teenage heaven\nI know, I know\nI'm a prisoner of hope, I know\n\nWe are young and we are old\nWe're fallin' like leaves\nAnd your heart's so full of soul\nIt makes me believe\nOnce again, it's something for you\n\nLucky love belongs in teenage heaven\nI know, I know\n'Cause I've been there with you tonight\nLucky love belongs in teenage heaven\nI know, I know\n'Cause I've been there with you tonight\n\nA bridge over time\nWas what you need to see the light

## Custom embeddings

In [20]:
def multi_stage_search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = qd_client.query_points(
        collection_name="test",
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="jinaai/jina-embeddings-v3",
                ),
                using="jina-v3",
                # Prefetch ten times more results, then
                # expected to return, so we can really rerank
                limit=(10 * limit),
            ),
        ],
        query=models.Document(
            text=query,
            model="Qdrant/bm25", 
        ),
        using="bm25",
        limit=limit,
        with_payload=True,
    )

    return results.points

In [36]:
result = multi_stage_search("song about hallucinations", limit = 3)

In [38]:
result[0]

ScoredPoint(id='2e2d113b-a551-444d-ac08-b2629b099de5', version=1, score=8.573651, payload={'lyrics': 'Mass hallucination, baby\nIll education, baby\nWant to reconnect with your elations?\nThis is your station, baby\n\nLook inside these walls\nAnd you see I\'m havin\' withdrawals of a prisoner on his way\nTrapped inside your desire to fire bullets that stray\nTrack attire just tell you I\'m tired and ran away\nI should ask a choir "What do you require\nTo sing a song that acquire me to have faith?"\n\nAs the record spin I should pray\nFor the record, I recognize that I\'m easily prey\nI got ate alive yesterday\nI got animosity buildin\', it\'s probably big as a buildin\'\nMe jumpin\' off of the roof is me just playin\' it safe\n\nBut what am I \'posed to do when the topic is red or blue\nAnd you understand that I ain\'t?\nBut know I\'m accustomed to just a couple that look for trouble\nAnd live in the street with rank\nNo better picture to paint than me walkin\' from bible study\nAnd ca

## RAG

In [9]:
from pydantic_ai import Agent, RunContext

In [10]:
developer_prompt="""
You are a music search assistant with access to a lyrics database.
Your goal is to help users find songs based on their descriptions, feelings, or vague memories.

## YOUR WORKFLOW

1. ANALYZE the user's query to understand what they're looking for
   (mood, theme, artist, era, imagery, storyline, etc.)

2. SEARCH using search_tool — up to 3 calls per turn, each with a different angle:
   - Rephrase the query to maximize semantic coverage
   - Try different aspects: emotional tone, specific imagery, narrative, artist style
   - Example: if user asks "sad Kanye love song", try:
       • "heartbreak regret relationship loss"
       • "Kanye West romantic melancholic introspective"
       • "love gone wrong self-reflection emotional"

3. EVALUATE results — after searching, check if any result genuinely matches 
   what the user described. Do NOT return a result just because it appeared 
   in search — verify it fits the description.

4. RESPOND to the user based on what you found.

## RESPONSE RULES

IF a confident match is found:
- State the song title and artist clearly
- Briefly explain WHY it matches the user's description (mood, theme, specific imagery)
- Keep it conversational and natural

IF no confident match is found:
- Be honest — do not hallucinate or guess song titles
- Tell the user what you searched for
- Ask 1 clarifying question to help narrow it down 
  (e.g., era, language, specific emotion, remembered fragment)

## CONSTRAINTS
- Maximum 3 search_tool calls per user message
- Never fabricate song titles or lyrics
- Never return a result you're not reasonably confident about
- Always prefer honest uncertainty over a wrong answer
""".strip()

In [11]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

model = OpenAIChatModel(
    model_name='openai/gpt-oss-20b',  # имя модели из `ollama list`
    provider=OllamaProvider(base_url='http://10.2.0.181:11434/v1')
)

In [12]:
chat_agent = Agent(  
    model,
    system_prompt=developer_prompt
)

In [15]:
from typing import Dict
import asyncio

@chat_agent.tool
async def search_tool(ctx: RunContext, query: str) -> Dict[str, str]:
    """
    Search the music lyrics database for relevant entries matching the query.

    Parameters
    ----------
    query : str
        The search query string provided by the user.

    Returns
    -------
    list
        A list of search results (up to 3), each containing relevance information 
        and associated output IDs.
    """
    print(f"search('{query}')")
    db_result = db.search(query, limit=3)
    
    result_parts = []
    for result in db_result:
        part = (
            f"Artist: {result.payload['artist']}, "
            f"Song Title: {result.payload['title']}\n"
            f"Song Lyrics:\n{result.payload['lyrics']}"
        )
        result_parts.append(part)
    
    return "\n\n".join(result_parts)

UserError: Tool name conflicts with existing tool: 'search_tool'

In [16]:
result_string

NameError: name 'result_string' is not defined

In [17]:
user_question = "Kanye song where he's struggling to move on from a toxic relationship and feels trapped in his emotion"
agent_run = await chat_agent.run(user_question)
print(agent_run.output)

Traceback (most recent call last):
  File "C:\Users\ivans\anaconda3\envs\lyrics\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ivans\AppData\Local\Temp\ipykernel_13720\4172828789.py", line 2, in <module>
    agent_run = await chat_agent.run(user_question)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ivans\anaconda3\envs\lyrics\Lib\site-packages\pydantic_ai\agent\abstract.py", line 392, in run
    node = await agent_run.next(node)  # pyright: ignore[reportArgumentType]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ivans\anaconda3\envs\lyrics\Lib\site-packages\pydantic_ai\run.py", line 380, in next
    return await self._run_node_with_hooks(node, self._advance_graph)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ivans\anaconda3\envs\lyrics\Lib\site-packages\pydantic_ai\run.py", line 306, in _run_node_with_hooks

In [58]:
user_question = "song with a fan writing letters not getting a reply goes insane"
lyrics = []
for result in db.search(user_question, title='Eminem', limit=3)[:5]:
    lyrics.append((result.payload['lyrics'], result.payload['artist'], result.payload['title']))

In [59]:
lyrics

[]

In [166]:
db.search(user_question, artist='Kanye West')[1]

IndexError: list index out of range

In [18]:
result_string = ""
user_question = 'Kasabian Club Foot'
db_result = db.search(user_question, limit=3)
for i, result in enumerate(db_result):
    result_string += f"Artist: {result.payload['artist']}, Song Title: {result.payload['title']}\nSong Lyrics:\n{result.payload['lyrics']}\n\n"

In [19]:
print(result_string)

Artist: Sia, Song Title: Miracle
Song Lyrics:
Nobody wants to wait for little miracles (little miracles)
Nobody wants to say, "I'm feeling insecure" (feeling insecure)
It's hard to be this big when I'm feeling small (I'm feeling small)
But I will keep on trying even when I fall

So, put one foot in front of the other
One foot in front of the other
We gotta love one another
We gotta love one another
One foot in front of the other
One foot in front of the other
We gotta love one another
We gotta love one another

I don't wanna quit, beforе the miracle (beforе the miracle)
I don't wanna quit, before the miracle (before the miracle)
I don't wanna quit, before the miracle (before the miracle)
I don't wanna quit, before the miracle (before the miracle)

Nobody wants to wait for little miracles (little miracles)
Nobody wants to say, "I'm feeling so unsure" (I'm feeling so unsure)
It's hard to be this big when I'm feelin' immature
But I'm gon' keep on tryin' even when I fall (even when I fall)

In [20]:
db_result[0]

ScoredPoint(id='5cccb584-6db7-475c-8779-98900a4bb9f6', version=146, score=0.5, payload={'lyrics': 'Nobody wants to wait for little miracles (little miracles)\nNobody wants to say, "I\'m feeling insecure" (feeling insecure)\nIt\'s hard to be this big when I\'m feeling small (I\'m feeling small)\nBut I will keep on trying even when I fall\n\nSo, put one foot in front of the other\nOne foot in front of the other\nWe gotta love one another\nWe gotta love one another\nOne foot in front of the other\nOne foot in front of the other\nWe gotta love one another\nWe gotta love one another\n\nI don\'t wanna quit, beforе the miracle (beforе the miracle)\nI don\'t wanna quit, before the miracle (before the miracle)\nI don\'t wanna quit, before the miracle (before the miracle)\nI don\'t wanna quit, before the miracle (before the miracle)\n\nNobody wants to wait for little miracles (little miracles)\nNobody wants to say, "I\'m feeling so unsure" (I\'m feeling so unsure)\nIt\'s hard to be this big when

In [33]:
qd_client.retrieve(ids=['a7b0e71f-c728-449e-8b09-5731292759d1'], collection_name="lyrics-hybrid")

[Record(id='a7b0e71f-c728-449e-8b09-5731292759d1', payload={'lyrics': "Procedamus in pace\nIn nomine Christi, Amen\n\nCum angelis et pueris\nFideles inveniamur\n\nAttollite portas principes vestras\nEt elevamini portae aeternales\nEt introibit rex gloriae\nQuis est iste Rex gloriae\n\nSade, dis-moi (dis-moi, dis-moi, dis-moi)\nSade donne-moi (donne-moi, donne-moi)\n\nProcedamus in pace\nIn nomine Christi, Amen\n\nSade, dis-moi\nQu'est-ce que tu vas chercher\nLe bien par le mal\nLa vertu par le vice\nSade, dis-moi pourquoi l'évangile du mal\nQuelle est ta religion, où sont tes fidèles\nSi tu es contre Dieu, tu es contre l'homme\nSade, es-tu diabolique ou divin\n\nSade, dis-moi (Hosanna)\nSade, donne-moi (Hosanna)\nSade, dis-moi (Hosanna)\nSade, donne-moi (Hosanna)\n\nIn nomine Christi, Amen", 'title': 'Sadeness, Part 1', 'artist': 'Enigma', 'album': 'The Platinum Collection', 'year': 2001, 'genre': 'Abstract', 'duration': 258}, vector=None, shard_key=None, order_value=None)]

## Creating questions

In [240]:
prompt = """
You are an expert question generator for evaluating RAG (Retrieval-Augmented Generation) systems 
that search over song lyrics.
Given:
- Artist: {artist}
- Song lyrics: {lyrics}
Generate exactly 1 question that simulates a REAL user trying to find this song.

QUESTION STYLE:
- Write as if a person vaguely remembers the song and is trying to find it
- Use natural, conversational language — not academic or formal
- Include the artist name in the question in 8 out of 10 cases
- Focus on UNIQUE, SPECIFIC details from THIS song — a particular story moment, 
  an unusual metaphor, a specific character, a concrete situation described in the lyrics
- DO NOT use vague emotional descriptions that could apply to hundreds of songs
  ("feeling lost", "longing for someone", "heartbreak") unless combined with 
  a specific concrete detail that anchors it to this song
- DO NOT mention the song title in the question
- DO NOT quote lyrics directly — paraphrase the imagery or situation instead

CRITICAL — LYRIC COVERAGE:
- DO NOT base the question on the opening lines or first verse of the song
- People often remember the chorus, a striking middle verse, the bridge, 
  or the outro — NOT the intro
- Mentally divide the lyrics into 4 parts (intro, early-middle, late-middle, ending) 
  and pick a detail from parts 2, 3, or 4 — NEVER from part 1
- Prefer details from the chorus, bridge, second/third verse, or final lines
- If the most memorable moment is genuinely in the intro, that's a rare exception — 
  but default to later sections

GOOD EXAMPLES:
- "Kanye song where he compares his ego to a deity but then breaks down 
   admitting he's terrified of being forgotten"
- "That Arctic Monkeys track where the guy is watching his ex at a party 
   through a window, smoking alone outside"
- "Kendrick song where he's literally having a court trial inside his own mind, 
   with different voices acting as judge and jury"

BAD EXAMPLES (too vague — could match hundreds of songs):
- "A song about missing someone you love with a haunting, repetitive melody"
- "Which The Weeknd song mentions drugs and love?" ← too generic
- "What song captures the feeling of being lost without someone?" ← no unique anchor

DIFFICULTY:
- The question must be specific enough to point to THIS song and no other
- Anchor it to a unique plot detail, image, or situation from the lyrics
- Abstract enough to require semantic search, not keyword matching

OUTPUT — strictly valid JSON, no markdown, no preamble:
{{
  "question": "...",
  "reasoning": "Brief explanation of which specific detail makes this question unique to this song, AND which section of the lyrics (chorus / verse 2 / bridge / outro / etc.) it comes from"
}}
""".strip()

In [22]:
with open ('metadata.json', 'r', encoding='utf-8') as f_in:
    data = json.load(f_in)

data_list = list(data.values())

In [4]:
for d in data_list[::100]:
    prepared_prompt = prompt.format(**d)
    break

NameError: name 'data_list' is not defined

In [195]:
from openai import OpenAI
from IPython.display import display, Markdown


client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # можно указать любую строку
)

In [196]:
def llm(query: str, system_prompt:str = None) -> str:

    if system_prompt:
        llm_query = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    else:
        llm_query = [
            {"role": "user", "content": query}
        ]
    
    response = client.chat.completions.create(
        model="llama3.2:3b",
        messages=llm_query,
        temperature=0.1,
        stream=False,  # или True для потокового вывода
        # extra_body={
        # "enable_thinking": False  # ← Отключает режим мышления
        # }
    )
    return response.choices[0].message.content

In [198]:
llm_q_a = []
for d in data_list[::100]:
    prepared_prompt = prompt.format(**d)
    llm_q_a.append(llm(prepared_prompt))

llm_q_a_list = [json.loads(x) for x in llm_q_a]

In [246]:
json.loads(llm(prepared_prompt))

JSONDecodeError: Expecting ',' delimiter: line 3 column 346 (char 546)

In [247]:
llm(prepared_prompt)

'{\n  "question": "Eminem song where he compares himself to a boy in a bubble who can\'t adapt and is trapped, and mentions his daughter Hailie",\n  "reasoning": "This question anchors on a unique plot detail (the \'boy in the bubble\' metaphor) from the late-middle section of the lyrics, making it specific enough to point to this song and no other. The mention of Hailie adds an additional layer of specificity, tying it back to Eminem\'s personal life and experiences."'

In [206]:
llm_q_a_list = [json.loads(x) for x in llm_q_a]

In [212]:
llm_q_a_list[30]

{'question': "Linkin Park song where the artist is trying to find their patience in a world that won't let them breathe",
 'reasoning': "This question references the specific line 'Lookin' for color in the black and white / Skyscrapers we created / On shaky ground' which is unique to this song, making it a distinctive query for semantic search."}

In [248]:
print(prepared_prompt)

You are an expert question generator for evaluating RAG (Retrieval-Augmented Generation) systems 
that search over song lyrics.
Given:
- Artist: Eminem
- Song lyrics: ♪
Sayin' goodbye, sayin' goodbye to Hollywood
Sayin' goodbye, sayin' goodbye to Hollywood
Sayin' goodbye, sayin' goodbye to Hollywood
Sayin' goodbye, sayin' goodbye to Hollywood
(Hollywood)
Sayin' goodbye, sayin' goodbye to Hollywood
(Why do I feel this way)
Sayin' goodbye, sayin' goodbye to Hollywood
Sayin' goodbye, sayin' goodbye to Hollywood
Sayin' goodbye, sayin' goodbye to Hollywood
I thought I had it all figured out, I did
I thought I was tough enough
to stick it out with Kim
But I wasn't tough enough
to juggle two things at once
I found myself layin' on my knees in cuffs
Which should've been a reason enough
for me to get my stuff and just leave
How come I couldn't see this shit myself
it's just me
Nobody couldn't see the shit I felt
Knowin' damn well she wasn't gonna be there when I fell
to catch me
The minute shit

In [243]:
prepared_prompt = prompt.format(**data['Eminem — Say Goodbye Hollywood'])

In [199]:
from collections import defaultdict

dd = defaultdict(int)
for d in data_list:
    dd[d['genre']]+=1

In [204]:
dd.keys()

dict_keys(['pop', None, 'Blues', 'Pop', 'Electronic', 'funk', 'Rock', 'Dance', 'Classic Rock', 'Hip Hop', 'Rap', 'Synthpop', 'R&B', 'Alternative Pop', 'Soul', 'Electronic, Pop', 'Pop Soul R&B', 'Dubstep', 'Hard Rock', 'New Wave', 'Electronic, Hip-Hop', 'Indie rock', 'Alternative Rock', 'Indie', 'Contemporary R&B', 'Поп', 'Disco', 'Miscellaneous', 'French House', 'Indie Rock', '5+ Wochen', 'Boogie', 'Funk', 'Alternative', 'House', 'Electro House', 'Progressive House', 'Indie Pop', 'Blues/Pop Rock', 'Soft Rock', 'Synth-pop', 'Electro', 'Aor', 'Dance-Pop', 'Album Rock', 'Pop, Miscellaneous', 'Hip-Hop', 'Abstract', 'Goth Rock', 'Soundtrack', 'Rap/Hip Hop', 'Alternative & Indie', 'Instrumental Pop', 'Eurodance', 'Club', 'Alternative Rock/Pop Rock', 'Acid Punk', 'Acid Jazz', 'East Coast Hip Hop', 'Rap, Nu-Metal', 'Alternative Hip Hop', 'Alternative Metal', 'Alt. Rock', 'Experimental', 'Dance-pop, Disco, Europop', 'Dance-pop, Nu-disco', 'Dance-Pop, Synthpop, Electroclash, R&B', 'Dance Pop, R&

## Create my RAG

In [3]:
db = LyricsDB(qdrant_client=qd_client, collection_name="jina-v2", model_name='jinaai/jina-embeddings-v2-small-en')
# db.fit(data)

Some weights of BertModel were not initialized from the model checkpoint at jinaai/jina-embeddings-v2-small-en and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.intermediate.dense.bias', 'encoder.layer.1.intermediate.dense.weight', 'encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.1.output.LayerNorm.weight', 'encoder.layer.1.output.dense.bias', 'encoder.layer.1.output.dense.weight', 'encoder.layer.2.intermediate.dense.bias', 'encoder.layer.2.intermediate.dense.weight', 'encoder.layer.2.output.LayerNorm.bias', 'encoder.layer.2.output.LayerNorm.weight', 'encoder.layer.2.output.dense.bias', 'encoder.layer.2.output.dense.weight', 'encoder.layer.3.intermediate.dense.bias', 'encoder.layer.3.intermediate.den

In [26]:
developer_prompt="""
You are a music search assistant with access to a lyrics database.
Your goal is to help users find songs based on their descriptions, feelings, or vague memories.

1. Analyze the existing CONTEXT from previously obtained song lyrics information searches:

<context>
{context}
</context>

2. Analyze the user's QUERY to understand what they're looking for (mood, theme, artist, era, imagery, storyline, etc.)

<query>
{query}
</query>

3. Based only on the CONTEXT above, give the answer to the user's QUERY. Analayze if the result fits the query, and provide the answer if it is.
If the context is empty or is not relevant, proceed further:

4. Create {num_queries} search QUERIES, which will contain the most relevant information from user's question in up to 10 words. 
Do not create same queries as they were in the provided context, try to rephrase the query.
QUERY MUST always contain not-empty value of "query" with your query, and optional with a "filters" values with 3 possible keys:
- "artist": if user's question obviously contain an artist name, provide it here. DO NOT include, if you are not sure
- "genre": if USER provides a genre (like "pop" or "rock" in the question) insert it here
- "year_range": if USER obsiously provide a song's year gap, insert it here. USE ONLY GAPS with 10 year starting from year last digit is 0 
(like 1980), to obtain a gap like 1980-1990. User's song year should be in the generated year range 



## RESPONSE RULES
Query should be responsed by the following JSON template:

{{
"query": provide your query here, 
"filters":{{
"artist": insert artist,
"genre": insert genre,
"year_range": insert year range from available}}
}}

DO NOT give conclusions based only on your internal knowledge base!
IF a confident match is found:
- State the song title and artist clearly
- Keep it conversational and natural

IF no confident match is found:
- Be honest — do not hallucinate or guess song titles
- Tell the user what you searched for
- Ask 1 clarifying question to help narrow it down 
  (e.g., era, language, specific emotion, remembered fragment)

## CONSTRAINTS
- Never fabricate song titles or lyrics
- Never return a result you're not reasonably confident about
- Always prefer honest uncertainty over a wrong answer
""".strip()

In [19]:
db.search('Mercedes CL', limit=3)

[ScoredPoint(id='edb450b8-2194-4b28-a31a-b63694545a81', version=71, score=0.5, payload={'lyrics': 'Boom, b-boom, b-boom-boom\n\nDrive slow, homie\nDrive slow, homie\nYou never know, homie\nMight meet some -, homie\nYou need to pump your brakes and drive slow, homie\n\nMy homie, Marley, used to stay, 79th and May\nOne of my best friends from back in the day\nDown the street from Calumet, a school full of stones\nHe nicknamed me K-Rock, so they\'d leave me alone\nBulls jacket with his hat broke way off\nAnd walked around the mall wit\' his radio face off\nPlus, he had the spinner from his Daytons in his hand\nKeys in his hand\nReason again to let you know he\'s the man\nBack when we rocked Ellesses\n\nHe had dreams of Caprices\nDrove by the teachers, even more by polices\nHow he get the cash the day his father passed away?\nLeft him wit\' a lil\' somethin\', 16 he was stuntin\'\nAl B. Sure! - with the hair all wavy\nHit Lake Shore, girls go all crazy\nHit the freeway, go at least \'bout 

In [21]:
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(
    base_url="http://10.2.0.181:11434/v1",
    api_key="ollama"  # можно указать любую строку
)

In [22]:
def llm(query: str, system_prompt:str = None) -> str:

    if system_prompt:
        llm_query = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    else:
        llm_query = [
            {"role": "user", "content": query}
        ]
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=llm_query,
        temperature=0.3,
        extra_body={
        "enable_thinking": False  # ← Отключает режим мышления
        }
    )
    return display(Markdown(response.choices[0].message.content))

In [27]:
query = "In which Kanye West song he sings about black Kate Moss"

In [28]:
llm(query, system_prompt=developer_prompt.format(context='', query=query, num_queries=2))

The lyric “black Kate Moss” appears in Kanye West’s song **“Runaway”** from his 2010 album *My Beautiful Dark Twisted Fantasy*.

```json
{
  "query": "Kanye West black Kate Moss lyric",
  "filters": {
    "artist": "Kanye West",
    "genre": "hip‑hop",
    "year_range": "2010-2020"
  }
}
```

```json
{
  "query": "Kanye West song about black Kate Moss",
  "filters": {
    "artist": "Kanye West",
    "genre": "hip‑hop",
    "year_range": "2010-2020"
  }
}
```

In [40]:
search = db.search('Kanye West song mentioning Black Kate Moss', limit=3, artist='Kanye West')

In [43]:
res_l = []
for s in search:
    res_l.append(s.payload['lyrics'])

In [44]:
res_l

['编曲 : Kanye West/Mitus/Metro Boomin/Noah Goldstein /Mike Dean\n制作人 : Kanye West/Mitus/Metro Boomin/Noah Goldstein/Mike Dean/Hudson Mohawke/Andrew Dawson\nProducers：Mike Dean/Metro Boomin/Noah Goldstein/Keyon Christ/Kanye West\nWriters：Hudson Mohawke/Keyon Christ/Darius "3D" Jenkins/K.Rachel Mills/Charlie Heat/Metro Boomin/Noah Goldstein Adrew Dawson/Mike Dean/Travis Scott/CyHi The Prynce/The Weeknd/Kanye West\nSample：Hit—Section 25\nI been waiting for a minute\nFor my lady\nSo I can\'t jeopardize that for one of these hoes\nI been living without limits\nAs far as my business\nI\'m the only one that\'s in control\nI been feeling all I\'ve given\nFor my children\nI will die for those I love\nGod, I\'m willing\nTo make this my mission\nGive up the women\nBefore I lose half of what I own\nI been thinking\nAbout my vision\nPour out my feelings\nRevealing the layers to my soul\nMy soul\nThe layers to my soul\nRevealing the layers to my soul\nThey wish I would go ahead and **** my life up\nC